# 🏎️ F1 Tire Analysis Notebook

Exploratory analysis of tire wear, degradation, and pit-stop strategy using the
`tire_agent` module and FastF1 data.

---

## Sections

1. **Setup** — imports, path configuration, FastF1 cache check  
2. **Track Wear Classification** — composite scores for all circuits, bar chart  
3. **Lap-Time Degradation** — per-compound degradation curve with regression lines  
4. **Pit Window Visualisation** — crossover model: cumulative deg loss vs pit-stop loss  
5. **Full Strategy Analysis** — `analyze_tire_strategy()` output vs actual race strategy  

---
## Section 1 — Setup

In [ ]:
# ── Path & environment ────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Add project root to sys.path so agent/tool imports resolve correctly
_NB_DIR = Path().resolve()
_PROJECT_ROOT = _NB_DIR.parent if _NB_DIR.name == 'notebooks' else _NB_DIR
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(_PROJECT_ROOT / '.env')

print(f'Project root : {_PROJECT_ROOT}')

In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import logging
import warnings

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Suppress noisy FastF1 / urllib3 warnings in notebook output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
logging.getLogger('fastf1').setLevel(logging.ERROR)
logging.getLogger('urllib3').setLevel(logging.ERROR)

print('Imports OK')

In [ ]:
# ── FastF1 cache check ────────────────────────────────────────────────────────
import fastf1

CACHE_DIR = _PROJECT_ROOT / 'data' / 'fastf1_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
fastf1.Cache.enable_cache(str(CACHE_DIR))

print(f'FastF1 version : {fastf1.__version__}')
print(f'Cache directory: {CACHE_DIR}')
print(f'Cache enabled  : {CACHE_DIR.exists()}')

In [ ]:
# ── Agent imports ─────────────────────────────────────────────────────────────
from agents.tire_agent import (
    classify_track_tire_wear,
    calculate_tire_degradation,
    estimate_pit_window,
    analyze_tire_strategy,
)
from tools.fastf1_tool import get_session, get_tire_data, get_stints, get_race_results

# ── Notebook-wide constants — change these to explore other races ─────────────
CIRCUIT      = 'Silverstone'
YEAR         = 2023
SESSION_TYPE = 'R'
DRIVER       = None   # None = race winner

# Pirelli compound colour palette (matches the Streamlit UI)
COMPOUND_COLORS = {
    'SOFT':         '#E8002D',
    'MEDIUM':       '#FFF200',
    'HARD':         '#FFFFFF',
    'INTERMEDIATE': '#39B54A',
    'WET':          '#0067FF',
    'UNKNOWN':      '#888888',
}

print(f'Analysis target: {CIRCUIT} {YEAR} {SESSION_TYPE}')

---
## Section 2 — Track Wear Classification

Compute the composite tire-wear score for every circuit in `track_features.csv`
and visualise results as a horizontal bar chart coloured by classification tier.

In [ ]:
# Load raw track features to get all circuit names
track_features_path = _PROJECT_ROOT / 'data' / 'raw' / 'track_features.csv'
track_df = pd.read_csv(track_features_path)

print(f'Circuits in database: {len(track_df)}')
print(f'Columns: {list(track_df.columns)}')
track_df.head()

In [ ]:
# Classify tire wear for every circuit
records = []
for track_name in track_df['TRACK'].tolist():
    result = classify_track_tire_wear(track_name)
    records.append({
        'Circuit':        result['track'],
        'Score':          result['score'],
        'Classification': result['classification'],
        'Tire Stress':    result['factors'].get('TIRE_STRESS', 0),
        'Lateral Load':   result['factors'].get('LATERAL', 0),
        'Asphalt Abr':    result['factors'].get('ASPHALT_ABR', 0),
        'Asphalt Grip':   result['factors'].get('ASPHALT_GRP', 0),
    })

wear_df = pd.DataFrame(records).sort_values('Score', ascending=False)
print(f'\nClassification breakdown:')
print(wear_df['Classification'].value_counts().to_string())
wear_df.head(10)

In [ ]:
# Detailed factor table
detail_df = wear_df[[
    'Circuit', 'Score', 'Classification',
    'Tire Stress', 'Lateral Load', 'Asphalt Abr', 'Asphalt Grip'
]].copy()

tier_emoji = {
    'High Tire Wear':   '🔴',
    'Medium Tire Wear': '🟡',
    'Low Tire Wear':    '🟢',
}
detail_df['Tier'] = detail_df['Classification'].map(tier_emoji)
detail_df[['Circuit', 'Tier', 'Score', 'Tire Stress', 'Lateral Load', 'Asphalt Abr', 'Asphalt Grip']]

In [ ]:
# Bar chart — composite score by circuit, coloured by tier
COLOR_MAP = {
    'High Tire Wear':   '#E8002D',
    'Medium Tire Wear': '#FFF200',
    'Low Tire Wear':    '#39B54A',
}

fig = px.bar(
    wear_df,
    x='Score',
    y='Circuit',
    color='Classification',
    color_discrete_map=COLOR_MAP,
    orientation='h',
    title='Track Tire Wear Classification — All Circuits',
    labels={'Score': 'Composite Wear Score (0–5)', 'Circuit': ''},
    template='plotly_dark',
    height=max(400, len(wear_df) * 22),
)

fig.update_layout(
    yaxis=dict(categoryorder='total ascending'),
    legend_title_text='Wear Tier',
    font=dict(family='Inter', size=12),
    margin=dict(l=130, r=40, t=60, b=40),
)

# Threshold lines
for threshold, label, color in [
    (2.0, 'Low/Medium boundary', '#aaaaaa'),
    (3.5, 'Medium/High boundary', '#ffffff'),
]:
    fig.add_vline(
        x=threshold, line_dash='dash', line_color=color, opacity=0.5,
        annotation_text=label,
        annotation_position='top',
        annotation_font_size=10,
    )

fig.show()

In [ ]:
# Stacked bar — factor contribution breakdown for top/bottom 10
factor_cols = ['Tire Stress', 'Lateral Load', 'Asphalt Abr', 'Asphalt Grip']
top_bottom = pd.concat([wear_df.head(8), wear_df.tail(5)]).drop_duplicates('Circuit')

fig2 = px.bar(
    top_bottom,
    x='Circuit',
    y=factor_cols,
    title='Factor Contribution to Wear Score (highest & lowest circuits)',
    labels={'value': 'Factor Score (0–5)', 'variable': 'Factor'},
    template='plotly_dark',
    barmode='group',
    color_discrete_sequence=['#E8002D', '#FFF200', '#39B54A', '#0067FF'],
)
fig2.update_layout(
    xaxis_tickangle=-40,
    font=dict(family='Inter', size=12),
    height=400,
    margin=dict(l=50, r=30, t=60, b=90),
)
fig2.show()

---
## Section 3 — Lap-Time Degradation

Load a live FastF1 race session and plot the lap-time degradation curve for
each stint, with linear regression lines coloured by compound.

In [ ]:
# Load FastF1 session (uses disk cache after first download)
print(f'Loading FastF1 session: {CIRCUIT} {YEAR} {SESSION_TYPE}...')
try:
    session = get_session(YEAR, CIRCUIT, SESSION_TYPE)
    results_df = get_race_results(session)
    if DRIVER is None:
        resolved_driver = results_df.iloc[0]['Abbreviation']
    else:
        resolved_driver = DRIVER
    print(f'Session loaded. Analysing driver: {resolved_driver}')
    print(f'Race winner: {results_df.iloc[0]["Abbreviation"]} ({results_df.iloc[0]["TeamName"]})')
except Exception as e:
    print(f'ERROR loading session: {e}')
    print('Hint: Check that FastF1 has data for this year/circuit combination.')
    raise

In [ ]:
# Fetch per-lap tire data for the resolved driver
tire_df = get_tire_data(session, driver=resolved_driver)
print(f'Lap records for {resolved_driver}: {len(tire_df)}')
print(f'Columns: {list(tire_df.columns)}')
tire_df.head()

In [ ]:
# Compute degradation rates via the tire agent
deg_data = calculate_tire_degradation(session, driver=resolved_driver)

print(f'Stints analysed: {len(deg_data)}')
deg_summary = pd.DataFrame(deg_data)[[
    'stint', 'compound', 'deg_rate_sec_per_lap',
    'avg_lap_sec', 'lap_count', 'start_lap', 'end_lap'
]]
deg_summary

In [ ]:
# Degradation scatter + regression lines
fig3 = go.Figure()

for stint_info in deg_data:
    stint_num = stint_info['stint']
    compound  = stint_info['compound']
    color     = COMPOUND_COLORS.get(compound, '#888888')
    deg_rate  = stint_info['deg_rate_sec_per_lap']

    # Raw data points for this stint
    stint_rows = tire_df[tire_df['Stint'] == stint_num].copy()
    if stint_rows.empty or 'LapTimeSec' not in stint_rows.columns:
        continue

    # Filter outliers (>110% of median)
    valid = stint_rows['LapTimeSec'].dropna()
    cutoff = valid.median() * 1.10
    clean = stint_rows[stint_rows['LapTimeSec'] <= cutoff]

    # Scatter plot (raw laps)
    fig3.add_trace(go.Scatter(
        x=clean['LapNumber'],
        y=clean['LapTimeSec'],
        mode='markers',
        name=f'Stint {stint_num} ({compound})',
        marker=dict(color=color, size=7, opacity=0.75,
                    line=dict(color='#333333', width=0.5)),
        legendgroup=f'stint{stint_num}',
    ))

    # Regression line
    if len(clean) >= 3:
        x_vals = clean['TyreLife'].values.astype(float)
        y_vals = clean['LapTimeSec'].values.astype(float)
        mask = np.isfinite(x_vals) & np.isfinite(y_vals)
        if mask.sum() >= 3 and np.std(x_vals[mask]) > 0:
            slope, intercept = np.polyfit(x_vals[mask], y_vals[mask], 1)
            x_line = np.linspace(x_vals[mask].min(), x_vals[mask].max(), 50)
            y_line = slope * x_line + intercept
            # Map TyreLife back to LapNumber for x-axis consistency
            lap_offset = clean['LapNumber'].min() - clean['TyreLife'].min()
            fig3.add_trace(go.Scatter(
                x=x_line + lap_offset,
                y=y_line,
                mode='lines',
                name=f'Stint {stint_num} trend ({deg_rate:+.3f} s/lap)',
                line=dict(color=color, width=2, dash='dot'),
                legendgroup=f'stint{stint_num}',
                showlegend=True,
            ))

fig3.update_layout(
    title=f'Lap-Time Degradation — {resolved_driver} · {CIRCUIT} {YEAR}',
    xaxis_title='Lap Number',
    yaxis_title='Lap Time (s)',
    template='plotly_dark',
    font=dict(family='Inter', size=12),
    height=480,
    margin=dict(l=60, r=30, t=70, b=50),
    legend=dict(orientation='v', x=1.02, y=1),
    hovermode='x unified',
)
fig3.show()

In [ ]:
# Degradation rate comparison across compounds (bar chart)
if deg_data:
    deg_bar_df = pd.DataFrame(deg_data)
    deg_bar_df['label'] = deg_bar_df.apply(
        lambda r: f"Stint {int(r['stint'])} ({r['compound']})", axis=1
    )
    deg_bar_df['color'] = deg_bar_df['compound'].map(
        lambda c: COMPOUND_COLORS.get(c, '#888888')
    )

    fig4 = go.Figure(go.Bar(
        x=deg_bar_df['label'],
        y=deg_bar_df['deg_rate_sec_per_lap'],
        marker_color=deg_bar_df['color'],
        text=deg_bar_df['deg_rate_sec_per_lap'].apply(lambda v: f'{v:+.4f} s/lap'),
        textposition='outside',
    ))
    fig4.update_layout(
        title=f'Degradation Rate per Stint — {resolved_driver} · {CIRCUIT} {YEAR}',
        xaxis_title='Stint (Compound)',
        yaxis_title='Deg Rate (s/lap)',
        template='plotly_dark',
        font=dict(family='Inter', size=12),
        height=350,
        margin=dict(l=60, r=30, t=60, b=60),
    )
    fig4.add_hline(y=0, line_dash='dash', line_color='#888888', opacity=0.5)
    fig4.show()

---
## Section 4 — Pit Window Visualisation

The crossover model compares the cumulative lap-time loss from tire degradation
against the one-off pit-stop time penalty.  The optimal pit lap is where the
degradation cost first equals (or exceeds) the pit-stop cost.

In [ ]:
# Compute pit window estimate
pit_window = estimate_pit_window(session, driver=resolved_driver)

print(f"Strategy type  : {pit_window['strategy_type']}")
print(f"Total laps     : {pit_window['total_laps']}")
print(f"Recommended pit laps: {pit_window['recommended_pit_laps']}")
print(f"\nRationale:\n{pit_window['rationale']}")

In [ ]:
# Pit window summary table
if pit_window['pit_windows']:
    pw_df = pd.DataFrame(pit_window['pit_windows'])
    pw_df.index = [f'Pit {i+1}' for i in range(len(pw_df))]
    pw_df.rename(columns={
        'earliest': 'Earliest Lap',
        'optimal':  'Optimal Lap',
        'latest':   'Latest Lap',
        'compound': 'Tyre',
        'deg_rate': 'Deg Rate (s/lap)',
    }, inplace=True)
    display(pw_df)
else:
    print('No pit windows computed (insufficient degradation data).')

In [ ]:
# Crossover visualisation: cumulative degradation cost vs pit-stop cost
PIT_LOSS_SEC = 22.0

if deg_data:
    fig5 = go.Figure()

    for i, stint in enumerate(deg_data):
        deg_rate = stint['deg_rate_sec_per_lap']
        if deg_rate <= 0:
            continue

        compound = stint['compound']
        color = COMPOUND_COLORS.get(compound, '#888888')
        max_laps = stint['end_lap'] - stint['start_lap'] + 1
        laps = np.arange(0, max_laps + 1)

        # Cumulative deg loss = deg_rate * n * (n+1) / 2  (triangular sum)
        cum_deg_loss = deg_rate * laps * (laps + 1) / 2

        fig5.add_trace(go.Scatter(
            x=laps + stint['start_lap'],
            y=cum_deg_loss,
            mode='lines',
            name=f'Stint {stint["stint"]} ({compound}) — cum. deg loss',
            line=dict(color=color, width=2),
        ))

    # Flat pit-stop penalty line
    total_laps = pit_window['total_laps']
    fig5.add_trace(go.Scatter(
        x=[1, total_laps],
        y=[PIT_LOSS_SEC, PIT_LOSS_SEC],
        mode='lines',
        name=f'Pit-stop cost ({PIT_LOSS_SEC} s)',
        line=dict(color='#ffffff', width=2, dash='dash'),
    ))

    # Optimal pit lap markers
    for pw in pit_window['pit_windows']:
        fig5.add_vline(
            x=pw['optimal'],
            line_dash='dot',
            line_color=COMPOUND_COLORS.get(pw['compound'], '#888'),
            opacity=0.7,
            annotation_text=f"Opt. L{pw['optimal']}",
            annotation_position='top',
            annotation_font_size=10,
        )

    fig5.update_layout(
        title=f'Pit Window Crossover Model — {resolved_driver} · {CIRCUIT} {YEAR}',
        xaxis_title='Race Lap',
        yaxis_title='Cumulative Time Loss (s)',
        template='plotly_dark',
        font=dict(family='Inter', size=12),
        height=420,
        margin=dict(l=60, r=30, t=70, b=50),
        hovermode='x unified',
    )
    fig5.show()
else:
    print('No degradation data available for crossover plot.')

---
## Section 5 — Full Strategy Analysis & Comparison

Run `analyze_tire_strategy()` and compare the model's compound recommendation
against the actual strategies used by the top-5 finishers.

In [ ]:
# Full analysis (re-uses the same session cache)
print(f'Running analyze_tire_strategy({CIRCUIT!r}, {YEAR}, {SESSION_TYPE!r})...')
analysis = analyze_tire_strategy(CIRCUIT, YEAR, SESSION_TYPE, DRIVER)

if analysis.get('error'):
    print(f'ERROR: {analysis["error"]}')
else:
    print('Analysis complete.')
    print(f'  Track wear  : {analysis["track_wear"]["classification"]} (score={analysis["track_wear"]["score"]})')
    print(f'  Strategy    : {analysis["pit_window"]["strategy_type"]}')
    print(f'  Rec. order  : {" → ".join(analysis["compound_rec"]["recommended_order"] or ["—"])}')
    if analysis['weather_impact']:
        wi = analysis['weather_impact']
        print(f'  Weather     : Air {wi["air_temp_avg"]}°C · Track {wi["track_temp_avg"]}°C · Rain={wi["rainfall"]}')

In [ ]:
# Actual strategies — top-5 finishers
stints_all = get_stints(session)
results_df = get_race_results(session)
top5 = list(results_df.head(5)['Abbreviation'])
top5_stints = stints_all[stints_all['Driver'].isin(top5)].copy()

# Build a clean strategy DataFrame for comparison
strategy_rows = []
for pos, drv in enumerate(top5, 1):
    drv_stints = top5_stints[top5_stints['Driver'] == drv].sort_values('Stint')
    if drv_stints.empty:
        continue
    compounds = drv_stints['Compound'].dropna().tolist()
    stops = len(compounds) - 1
    team = results_df[results_df['Abbreviation'] == drv].iloc[0].get('TeamName', '')
    strategy_rows.append({
        'Pos':      pos,
        'Driver':   drv,
        'Team':     team,
        'Stops':    stops,
        'Strategy': ' → '.join(compounds),
        'Compounds': compounds,
    })

strat_df = pd.DataFrame(strategy_rows)
print(f'Top-5 actual strategies ({CIRCUIT} {YEAR}):')
strat_df[['Pos', 'Driver', 'Team', 'Stops', 'Strategy']]

In [ ]:
# Model recommendation vs actual winner strategy
rec   = analysis.get('compound_rec') or {}
model_strategy = rec.get('recommended_order', [])
model_stops    = max(0, len(model_strategy) - 1)

winner_row    = strat_df[strat_df['Pos'] == 1].iloc[0]
actual_strat  = winner_row['Compounds']
actual_stops  = winner_row['Stops']

print('=' * 55)
print(f'  Comparison: {CIRCUIT} {YEAR} — {winner_row["Driver"]}')
print('=' * 55)
print(f'  Model recommendation : {" → ".join(model_strategy)} ({model_stops}-stop)')
print(f'  Actual winner        : {" → ".join(actual_strat)} ({actual_stops}-stop)')
print()

# Overlap analysis
model_set  = set(model_strategy)
actual_set = set(actual_strat)
overlap    = model_set & actual_set
match_pct  = 100 * len(overlap) / max(len(actual_set), 1)

print(f'  Compound overlap     : {overlap} ({match_pct:.0f}% match)')
print(f'  Stop count match     : {"Yes" if model_stops == actual_stops else "No"}')

In [ ]:
# Stint timeline for top-5 (Gantt-style)
if not strat_df.empty:
    timeline_rows = []
    for _, row in strat_df.iterrows():
        drv = row['Driver']
        drv_stints = top5_stints[top5_stints['Driver'] == drv].sort_values('Stint')
        for _, s in drv_stints.iterrows():
            compound = s.get('Compound', 'UNKNOWN')
            start    = int(s.get('StartLap', 0))
            end      = int(s.get('EndLap', start))
            timeline_rows.append({
                'Driver':   drv,
                'Compound': compound,
                'Start':    start,
                'End':      end,
                'Laps':     end - start + 1,
            })

    tl_df = pd.DataFrame(timeline_rows)

    fig6 = go.Figure()
    for _, row in tl_df.iterrows():
        color = COMPOUND_COLORS.get(row['Compound'], '#888888')
        fig6.add_trace(go.Bar(
            x=[row['Laps']],
            y=[row['Driver']],
            base=[row['Start']],
            orientation='h',
            marker_color=color,
            marker_line=dict(color='#1a1a2e', width=1),
            name=row['Compound'],
            text=row['Compound'],
            textposition='inside',
            textfont=dict(color='#1a1a2e', size=10),
            legendgroup=row['Compound'],
            showlegend=(row['Driver'] == tl_df['Driver'].iloc[0]),
            hovertemplate=(
                f"{row['Driver']}: {row['Compound']}<br>"
                f"Laps {row['Start']}–{row['End']} ({row['Laps']} laps)<extra></extra>"
            ),
        ))

    fig6.update_layout(
        barmode='stack',
        title=f'Stint Strategy Timeline — Top-5 Finishers · {CIRCUIT} {YEAR}',
        xaxis_title='Race Lap',
        yaxis_title='Driver',
        template='plotly_dark',
        font=dict(family='Inter', size=12),
        height=320,
        margin=dict(l=60, r=30, t=70, b=50),
        yaxis=dict(categoryorder='array', categoryarray=top5[::-1]),
    )
    fig6.show()

In [ ]:
# Compound distribution per driver (stacked bar)
if not strat_df.empty:
    comp_rows = []
    for _, row in strat_df.iterrows():
        from collections import Counter
        for compound, count in Counter(row['Compounds']).items():
            comp_rows.append({'Driver': row['Driver'], 'Compound': compound, 'Stints': count})
    comp_df = pd.DataFrame(comp_rows)

    fig7 = px.bar(
        comp_df,
        x='Driver',
        y='Stints',
        color='Compound',
        color_discrete_map=COMPOUND_COLORS,
        title=f'Compound Usage Distribution — {CIRCUIT} {YEAR}',
        template='plotly_dark',
        barmode='stack',
        height=350,
    )
    fig7.update_layout(
        font=dict(family='Inter', size=12),
        margin=dict(l=50, r=30, t=60, b=50),
        yaxis_title='Number of Stints',
    )
    fig7.show()

In [ ]:
# Summary table: model outputs side-by-side
summary = {
    'Circuit':            CIRCUIT,
    'Year':               YEAR,
    'Driver Analysed':    resolved_driver,
    'Track Wear':         analysis['track_wear']['classification'],
    'Wear Score':         analysis['track_wear']['score'],
    'Strategy Type':      analysis['pit_window']['strategy_type'],
    'Recommended Compounds': ' → '.join(model_strategy) if model_strategy else '—',
    'Optimal Pit Laps':   str(pit_window['recommended_pit_laps']),
    'Winner Strategy':    ' → '.join(actual_strat),
    'Stop Count Match':   'Yes' if model_stops == actual_stops else 'No',
    'Compound Overlap %': f'{match_pct:.0f}%',
}
if analysis['weather_impact']:
    wi = analysis['weather_impact']
    summary['Air Temp Avg']   = f"{wi['air_temp_avg']} °C"
    summary['Track Temp Avg'] = f"{wi['track_temp_avg']} °C"
    summary['Rainfall']       = str(wi['rainfall'])

pd.DataFrame.from_dict(summary, orient='index', columns=['Value'])

---
## Next Steps

- Change `CIRCUIT`, `YEAR`, or `DRIVER` in **Section 1** to explore another race.
- Run `python -m agents.tire_agent` from the project root for a CLI summary.
- Use `analyze_tire_strategy()` output as input to `agents/strategist_agent.py`
  for a full LLM-powered recommendation.
- Inspect `rag/retriever.py` → `retrieve_race_context()` to combine tire data
  with historical race strategy text for richer LLM context.